# Mattis EDA

Initial loading and first overview inspection for data in `data/raw/Alternative Medien/`.


In [41]:
from pathlib import Path
import pandas as pd

In [42]:
# Set Path
PROJECT_ROOT = Path.cwd().parent
BASE_DIR = PROJECT_ROOT / "data" / "raw" / "Alternative Medien"

print("cwd:", Path.cwd())
print("PROJECT_ROOT:", PROJECT_ROOT)
print("BASE_DIR:", BASE_DIR)
print("exists:", BASE_DIR.exists())

if not BASE_DIR.exists():
    raise FileNotFoundError(f"BASE_DIR not found: {BASE_DIR}")


cwd: /Users/MattisHaumann/Dev/Thesis/Initial EDA
PROJECT_ROOT: /Users/MattisHaumann/Dev/Thesis
BASE_DIR: /Users/MattisHaumann/Dev/Thesis/data/raw/Alternative Medien
exists: True


In [43]:
# CSV reader (for every source except RT)
def read_csv_resilient(csv_path: Path) -> pd.DataFrame:
    encodings = ("utf-8", "utf-8-sig", "latin-1")

    # normal read
    for enc in encodings:
        try:
            return pd.read_csv(
                csv_path,
                encoding=enc,
                low_memory=False,
                on_bad_lines="skip",
            )
        except Exception:
            pass

    # delimiter sniff (python engine)
    for enc in encodings:
        try:
            return pd.read_csv(
                csv_path,
                encoding=enc,
                sep=None,
                engine="python",
                low_memory=False,
                on_bad_lines="skip",
            )
        except Exception:
            pass

    raise ValueError(f"Could not read: {csv_path}")

In [44]:
dfs_by_source = {}
overview_rows = []

source_dirs = sorted(
    [p for p in BASE_DIR.iterdir() if p.is_dir()],
    key=lambda p: p.name.lower()
)

for source_dir in source_dirs:
    csv_files = sorted(source_dir.rglob("*.csv"))
    frames = []
    failed = 0

    for csv_file in csv_files:
        try:
            part = read_csv_resilient(csv_file)
            part["source"] = source_dir.name
            part["source_file"] = csv_file.name  # keep it simple
            frames.append(part)
        except Exception as e:
            failed += 1
            print(f"[WARN] {source_dir.name} -> {csv_file.name}: {e}")

    df_source = pd.concat(frames, ignore_index=True, sort=False) if frames else pd.DataFrame()
    dfs_by_source[source_dir.name] = df_source

    overview_rows.append({
        "source": source_dir.name,
        "files_found": len(csv_files),
        "files_failed": failed,
        "total_articles": len(df_source),
        "n_columns": df_source.shape[1],
    })

# RT_de.xlsx at the root
rt_path = BASE_DIR / "RT_de.xlsx"
if rt_path.exists():
    df_rt = pd.read_excel(rt_path)
    df_rt["source"] = "RT_de"
    df_rt["source_file"] = rt_path.name
    dfs_by_source["RT_de"] = df_rt

    overview_rows.append({
        "source": "RT_de",
        "files_found": 1,
        "files_failed": 0,
        "total_articles": len(df_rt),
        "n_columns": df_rt.shape[1],
    })

overview_df = (
    pd.DataFrame(overview_rows)
    .sort_values(["total_articles", "source"], ascending=[False, True])
    .reset_index(drop=True)
)

display(overview_df)


,source,files_found,files_failed,total_articles,n_columns
0,deusch_pravda,496,0,397944,9
1,RT_de,1,0,12390,9
2,Nius_Rohdaten_neu,285,0,4885,29
3,apollo,165,0,3550,7
4,Tichy's Einblick,195,0,3126,8
5,Compact,214,0,2045,6
6,Antispiegel,307,0,914,9
7,Reitschuster,238,0,685,6


In [45]:
src = "deusch_pravda"

if src in dfs_by_source:
    df = dfs_by_source[src]

    if "time" in df.columns:
        # Parse German datetime and drop time part
        parsed = pd.to_datetime(
            df["time"],
            format="%d.%m.%Y, %H:%M",
            errors="coerce"
        )

        df["date"] = parsed.dt.normalize()  # keeps only YYYY-MM-DD

        dfs_by_source[src] = df

        print(f"[OK] {src}: created clean 'date' column (without time)")
    else:
        print(f"[WARN] {src}: no 'time' column found")


[OK] deusch_pravda: created clean 'date' column (without time)


In [46]:
DATE_CANDIDATES = ["Date", "date", "day", "published", "published_at", "datetime", "timestamp"]

date_rows = []

for source, df in dfs_by_source.items():
    date_min = pd.NaT
    date_max = pd.NaT

    for col in DATE_CANDIDATES:
        if col in df.columns:
            s = pd.to_datetime(df[col], errors="coerce").dt.normalize()
            if s.notna().any():
                date_min = s.min()
                date_max = s.max()
                break

    date_rows.append({
        "source": source,
        "date_min": date_min,
        "date_max": date_max,
    })

date_df = pd.DataFrame(date_rows)

overview_df = overview_df.drop(columns=["date_min", "date_max"], errors="ignore")
overview_df = overview_df.merge(date_df, on="source", how="left")

display(overview_df)


,source,files_found,files_failed,total_articles,n_columns,date_min,date_max
0,deusch_pravda,496,0,397944,9,2025-05-05,2025-09-02
1,RT_de,1,0,12390,9,2024-11-14,2026-01-19
2,Nius_Rohdaten_neu,285,0,4885,29,2025-05-01,2026-02-10
3,apollo,165,0,3550,7,2025-08-12,2026-02-10
4,Tichy's Einblick,195,0,3126,8,2025-08-01,2026-02-11
5,Compact,214,0,2045,6,2025-06-11,2026-02-10
6,Antispiegel,307,0,914,9,2025-04-10,2026-02-10
7,Reitschuster,238,0,685,6,2025-06-10,2026-02-10


# Articles per month

In [47]:
DATE_CANDIDATES = ["Date", "date", "day", "published", "published_at", "datetime", "timestamp"]

monthly_rows = []

for source, df in dfs_by_source.items():
    used_series = None

    # find a usable date column
    for col in DATE_CANDIDATES:
        if col in df.columns:
            s = pd.to_datetime(df[col], errors="coerce").dt.normalize()
            if s.notna().any():
                used_series = s
                break

    if used_series is None:
        continue

    # convert to year-month period
    ym = used_series.dt.to_period("M").astype(str)

    # count articles per month
    counts = ym.value_counts()

    for month, count in counts.items():
        monthly_rows.append({"source": source, "year_month": month, "articles": count})

monthly_df = pd.DataFrame(monthly_rows)

pivot = (
    monthly_df
    .pivot_table(index="source", columns="year_month", values="articles", aggfunc="sum", fill_value=0)
    .sort_index()
)

# optional: sort columns chronologically (YYYY-MM already sorts correctly as strings)
pivot = pivot.reindex(sorted(pivot.columns), axis=1)

display(pivot)


year_month,2024-11,2024-12,2025-01,2025-02,2025-03,2025-04,2025-05,2025-06,2025-07,2025-08,2025-09,2025-10,2025-11,2025-12,2026-01,2026-02
source,,,,,,,,,,,,,,,,
Antispiegel,0,0,0,0,0,85,118,34,32,92,122,113,85,114,87,32
Compact,0,0,0,0,0,0,0,168,176,207,275,301,280,258,293,87
Nius_Rohdaten_neu,0,0,0,0,0,0,482,447,495,466,563,589,604,509,548,182
RT_de,522,1031,958,939,941,862,895,785,887,687,856,855,898,868,406,0
Reitschuster,0,0,0,0,0,0,0,65,106,72,100,0,88,0,67,22
Tichy's Einblick,0,0,0,0,0,0,0,0,0,458,486,510,483,504,505,180
apollo,0,0,0,0,0,0,0,0,0,176,565,647,643,635,646,238
deusch_pravda,0,0,0,0,0,0,22016,30688,39674,36574,2249,0,0,0,0,0


## Erste Insights
Klare Unterschiede zwischen den Quellen.

- **deusch_pravda** hat extrem hohe Zahlen, aber nur in wenigen Monaten (Mai–September 2025). Sind wahrscheinlich Telegramdaten und nicht klassisch news

- **RT_de** veröffentlicht konstant über einen langen Zeitraum (Ende 2024–Anfang 2026). Sehr strukturiert und institutionell

- **Nius, Apollo, Compact und Tichys Einblick** zeigen relativ regelmäßige monatliche Output-Zahlen. Auch stabile redaktionelle Routinen vermutlich

- **Antispiegel und Reitschuster** haben deutlich geringeres Volumen. Vermutlich kleinere Strukturen oder stärker meinungsgetriebene Formate.

**Fazit (erste EDA):**  
Schon auf dieser deskriptiven Ebene sieht man, dass die untersuchten „alternativen Medien“ sehr unterschiedlich arbeiten – sowohl im Volumen als auch in der zeitlichen Dynamik.


# Length of Articles

In [48]:
# map using lowercase source names to avoid case issues
TEXT_COLUMN_MAP = {
    "antispiegel": "Full_Text",
    "apollo": "Inhalt",
    "compact": "Inhalt",
    "deusch_pravda": "full_text",
    "reitschuster": "Inhalt",
    "nius_rohdaten_neu": "article_text",
    "rt_de": "Full_Text",
    "tichy's einblick": "article_text",
}

length_rows = []

for source, df in dfs_by_source.items():
    key = source.lower()  # normalize
    text_col = TEXT_COLUMN_MAP.get(key)

    if text_col is None or text_col not in df.columns:
        print(f"[WARN] {source}: text column not found or not mapped. (mapped={text_col})")
        continue

    words = df[text_col].astype(str).str.split().str.len()

    length_rows.append({
        "source": source,
        "articles": len(df),
        "text_col": text_col,
        "mean_words": words.mean(),
        "median_words": words.median(),
        "min_words": words.min(),
        "max_words": words.max(),
    })

length_df = (
    pd.DataFrame(length_rows)
    .sort_values("mean_words", ascending=False)
    .reset_index(drop=True)
)

display(length_df)

,source,articles,text_col,mean_words,median_words,min_words,max_words
0,Antispiegel,914,Full_Text,2203.440919,1720.0,170.0,18894.0
1,Reitschuster,685,Inhalt,1584.382482,1466.0,837.0,5014.0
2,Tichy's Einblick,3126,article_text,827.087332,765.0,42.0,5857.0
3,RT_de,12390,Full_Text,629.242696,470.5,7.0,4689.0
4,Nius_Rohdaten_neu,4885,article_text,586.183498,426.0,6.0,64721.0
5,Compact,2045,Inhalt,504.015159,417.0,23.0,4324.0
6,apollo,3550,Inhalt,444.925634,384.0,23.0,2662.0
7,deusch_pravda,397944,full_text,162.234063,114.0,1.0,7820.0


## Erste Insights 

Klare strukturelle Unterschiede zwischen den Quellen.

- **deusch_pravda** hat extrem hohe Volumina in kurzer Zeit. Gleichzeitig ist die durchschnittliche Textlänge deutlich niedriger. Das spricht für aggregierte Inhalte (z. B. Telegram) statt klassischer redaktioneller Langform.

- **RT_de** sowie **Antispiegel** und **Reitschuster** weisen im Schnitt deutlich längere Texte auf. Das deutet eher auf ausführlichere Beiträge oder kommentierende Formate hin.

- **Nius, Apollo, Compact und Tichys Einblick** liegen im mittleren Bereich. Hier sieht man eher typische Online-Artikel-Längen.

Wichtig:  
Die Textdaten sind teilweise noch stark „roh“. Es finden sich z. B. Werbeeinblendungen, wiederholte Textpassagen oder technische Artefakte. Vor tiefergehenden inhaltlichen Analysen (z. B. Sentiment, Topic Modeling) ist daher eine systematische Textbereinigung notwendig.

**Zwischenfazit:**  
Schon in dieser frühen EDA zeigen sich klare Unterschiede in Produktionslogik, Umfang und Struktur der Inhalte. Die Datenqualität und Textbereinigung werden dabei ein zentraler methodischer Schritt für die weitere Analyse sein.


## Dateninspektion pro Quelle

In [49]:
import random

SAMPLE_SIZE = 5

for source, df in dfs_by_source.items():
    print("\n" + "="*80)
    print(f"Source: {source}")
    print("="*80)

    if len(df) == 0:
        print("No data available.")
        continue

    sample_df = df.sample(min(SAMPLE_SIZE, len(df)), random_state=42)

    for i, row in sample_df.iterrows():
        print("\n--- Article ---")

        # Title if available
        if "title" in df.columns:
            print("Title:", row.get("title"))
        elif "Title" in df.columns:
            print("Title:", row.get("Title"))

        # Try mapped text column if already defined
        text_col = None
        for col in df.columns:
            if col.lower() in ["full_text", "text", "inhalt", "article_text"]:
                text_col = col
                break

        if text_col:
            text_preview = str(row[text_col])[:1000]
            print("\nText Preview:\n", text_preview)
        else:
            print("No obvious text column found.")



Source: Antispiegel

--- Article ---
Title: Beginnt jetzt die „GPS-Show“?

Text Preview:
 Propaganda
Beginnt jetzt die „GPS-Show“?
Schweden meldet eine große Zunahme der Störungen des GPS in der zivilen Luftfahrt. Schuld ist angeblich natürlich Russland und als Beleg erinnert der Spiegel an die Störung des Fluges von von der Leyen, die sich jedoch als Fake herausgestellt hat. Die westliche Propaganda lügt immer lustiger.
von Anti-Spiegel
6. September 2025 12:00 Uhr
Als am Montag gemeldet wurde, das GPS des Fluges von EU-Kommissionschefin Ursula von der Leyen von Warschau ins bulgarische Plowdiw sei von Russland gestört worden und die Maschine hätte deswegen eine Stunde lang Warteschleifen fliegen müssen, bevor die Piloten landen konnten, war schnell klar, dass die Geschichte frei erfunden war.
Die GPS-Lüge vom Montag
Erstens liegt Plowdiw etwa 200 Kilometer vom Schwarzen Meer entfernt, sodass, wenn Russland das GPS gestört hätte, das GPS im halben Bulgarien hätte gestört sein müssen. 

In [50]:
# Zeige das DataFrame für Russia Today (RT_de)
if "RT_de" in dfs_by_source:
    rt_df = dfs_by_source["RT_de"]
elif "df_rt" in globals():
    rt_df = df_rt
else:
    raise KeyError("RT_de nicht gefunden (weder in dfs_by_source noch als df_rt)")

print("RT_de shape:", rt_df.shape)
display(rt_df)

RT_de shape: (12390, 9)


Date                        Category  \
0      2026-01-19    Hauptseite\n/\nInternational   
1      2026-01-19    Hauptseite\n/\nInternational   
2      2026-01-19  Hauptseite\n/\nNahost-Konflikt   
3      2026-01-19          Hauptseite\n/\nSchweiz   
4      2026-01-19    Hauptseite\n/\nInternational   
...           ...                             ...   
12385  2024-11-14          Hauptseite\n/\nSchweiz   
12386  2024-11-14    Hauptseite\n/\nUkraine-Krieg   
12387  2024-11-14       Hauptseite\n/\nÖsterreich   
12388  2024-11-14      Hauptseite\n/\nDeutschland   
12389  2024-11-14      Hauptseite\n/\nDeutschland   

                                                                                                   Title  \
0                                Trump: Dänemark kann "russische Bedrohung" in Grönland nicht beseitigen   
1                                                  Mindestens 39 Tote bei schwerem Zugunglück in Spanien   
2              Syriens Machthaber al-Scharaa verschiebt wegen Militäroperationen seinen Termin in Berlin   
3                            Davos wird zum US-Forum: Zum Auftakt steht das WEF im Zeichen Donald Trumps   
4      Welch erbärmliche und prinzipienlose Kreaturen: Warum die Europäer nun Schutz bei Russland suchen   
...                                                                                                  ...   
12385                            Gedemütigt und geschlagen: Was Prostituierte in der Schweiz durchmachen   
12386                                    Bloomberg: Ukrainer spenden immer weniger für ihre Streitkräfte   
12387                                                Österreich: Traditionsmarke Kika/Leiner vor dem Aus   
12388                                                 Sachsen: CDU und SPD wollen jetzt allein koalieren   
12389                            Falls Mützenich Außenminister wird: Andrei Melnyk kündigt Selbstmord an   

                                                                                                                                                                                                                                                                                                                                                                 Summary  \
0                                                                                                           Die Nordatlantikallianz habe Dänemark 20 Jahre lang gebeten, die "russische Bedrohung" für Grönland zu beseitigen, aber Kopenhagen habe diese Aufgabe nicht bewältigt, schrieb US-Präsident Donald Trump. Jetzt sei es "an der Zeit", diese Frage zu klären.   
1                                                                                                      Bei einem schweren Zugunglück nahe Córdoba entgleisten und kollidierten zwei Hochgeschwindigkeitszüge. Mindestens 39 Menschen kamen ums Leben, rund 70 wurden verletzt. Die Ursache ist unklar. Der Bahnverkehr zwischen Madrid und Andalusien bleibt ausgesetzt.   
2                      Der für Anfang der Woche angekündigte Besuch des syrischen Übergangspräsidenten Ahmed al-Scharaa im Berliner Kanzleramt ist seitens Damaskus abgesagt worden. Grund seien die militärischen Ereignisse im Nordosten des Landes. Syrische Medien haben bekannt gegeben, dass mit den kurdischen SDF-Kräften ein Waffenstillstand vereinbart wurde.   
3                                                                                                                                                  Die größte US-Delegation aller Zeiten, massive Sicherheitsvorkehrungen und konkrete Machtpolitik machen Davos zur Bühne amerikanischer Interessen. Noch vor der Eröffnung steht fest: Dieses WEF gehört Donald Trump.   
4                                                                       Europa hat Russlands Rolle verkannt und zahlt nun den Preis, während Washington das Völkerrecht und die UNO offen entwertet. Der geplante "Friedensrat" der USA verstärkt die Angst vor ein

In [51]:
# RT today sample article + link to click

from IPython.display import HTML
from html import escape

pd.set_option("display.max_colwidth", None)

# get RT_de frame
if "RT_de" in dfs_by_source:
    rt = dfs_by_source["RT_de"]
elif "rt_df" in globals():
    rt = rt_df
else:
    raise KeyError("RT_de not found in dfs_by_source or as rt_df")

cols_map = {c.lower(): c for c in rt.columns}
full_col = cols_map.get("full_text") or cols_map.get("text") or cols_map.get("fulltext")
url_col  = cols_map.get("url") or cols_map.get("link")

if not full_col or not url_col:
    raise KeyError(f"Could not find text/url columns in RT_de. Available columns: {list(rt.columns)}")

sample_df = rt[[full_col, url_col]].sample(5, random_state=42).reset_index(drop=True)

def format_text(t):
    t = escape(str(t)).replace("\n", "<br>")
    return "<div style='max-width:1200px;white-space:normal;overflow:auto'>" + t + "</div>"

sample_df["full_text"] = sample_df[full_col].apply(format_text)
sample_df["url"] = sample_df[url_col].astype(str).apply(
    lambda u: f'<a href="{escape(u)}" target="_blank" rel="noopener noreferrer">{escape(u)}</a>'
)

html = sample_df[["full_text", "url"]].to_html(escape=False, index=False)
display(HTML(html))


## Topic Modelling (RT_de) — Preprocessing & Rationale

**Data input (RT_de)**
- Dataframe: `rt_topic_df` from `dfs_by_source["RT_de"]` (fallback: `rt_df`).
- Columns used (explicit): `Full_Text` for text and `Date` for timestamps.

**Why minimal preprocessing here?**
- `Full_Text` is already relatively clean, so we keep preprocessing simple and focused on stopword removal for better topic quality.

**Cleaning steps**
1. Copy raw model text into `topic_text_raw` (source columns are not overwritten).
2. Minimal normalization: lowercasing, URL removal, whitespace normalization.
3. Stopword removal: spaCy German stopwords (`spacy.lang.de.stop_words.STOP_WORDS`) + short custom list (`rt`, `https`, `http`, `www`, `de`, `freedert`, `online`).
4. Token filtering: remove numeric-only tokens and very short tokens (`< 2` chars).

**Chunking strategy (implemented)**
- We chunk each cleaned article into fixed token windows before BERTopic.
- Settings: `CHUNK_SIZE_TOKENS = 220`, `CHUNK_OVERLAP_TOKENS = 40`, `MIN_CHUNK_TOKENS = 40`.
- Why: long articles can dominate embeddings; chunking keeps topic signals more balanced and still preserves local context via overlap.

**Reproducibility notes**
- Deterministic rules and fixed embedding model: `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`.
- BERTopic settings fixed (`language="multilingual"`, `min_topic_size=25`, `calculate_probabilities=False`).
- Default uses full RT_de (`MAX_DOCS = None`); optional sampling uses `random_state=42`.

**Sanity checks**
- Print article/chunk counts and date range used for BERTopic.
- Show top tokens after cleaning to verify stopword removal worked (no `der/die/das` dominance).


In [52]:
%pip install spacy

Note: you may need to restart the kernel to use updated packages.


In [53]:
# BERTopic auf RT_de (clean + reproducible)
# Optional one-time install (if needed):
# %pip install bertopic sentence-transformers umap-learn hdbscan huggingface_hub spacy

import os
import re
from collections import Counter

import pandas as pd

try:
    from bertopic import BERTopic
except ImportError as e:
    raise ImportError(
        "BERTopic not installed. Run: %pip install bertopic sentence-transformers umap-learn hdbscan"
    ) from e

# Optional HF auth via environment only (no interactive prompt)
hf_token = os.getenv('HF_TOKEN')
if hf_token:
    try:
        from huggingface_hub import login
        login(token=hf_token, add_to_git_credential=False, skip_if_logged_in=True)
        print('Hugging Face login via HF_TOKEN: active')
    except Exception as e:
        print(f'Hugging Face login skipped: {e}')
else:
    print('HF_TOKEN not set. Continuing without explicit HF login.')

# 1) Get RT_de dataframe from existing EDA objects
if 'RT_de' in dfs_by_source:
    rt_topic_df = dfs_by_source['RT_de'].copy()
elif 'rt_df' in globals():
    rt_topic_df = rt_df.copy()
else:
    raise KeyError('RT_de not found in dfs_by_source or rt_df')

# 2) Use explicit RT_de columns from your schema
required_cols = {'Full_Text', 'Date'}
missing_cols = required_cols.difference(rt_topic_df.columns)
if missing_cols:
    raise KeyError(f'Missing required RT_de columns: {sorted(missing_cols)}')

# 3) Create topic-modelling columns (keep original columns unchanged)
rt_topic_df['topic_text_raw'] = rt_topic_df['Full_Text'].astype(str)
rt_topic_df['topic_date'] = pd.to_datetime(rt_topic_df['Date'], errors='coerce')

# 4) Stopwords: spaCy German stopwords + short custom additions
try:
    from spacy.lang.de.stop_words import STOP_WORDS
except ImportError as e:
    raise ImportError(
        "spaCy German stopwords are required. Run: %pip install spacy"
    ) from e

german_stopwords = set(STOP_WORDS)
custom_stopwords = {'rt', 'https', 'http', 'www', 'de', 'freedert', 'online'}
all_stopwords = german_stopwords.union(custom_stopwords)

url_pattern = re.compile(r'https?://\S+|www\.\S+', flags=re.IGNORECASE)
token_pattern = re.compile(r'[^\W_]+', flags=re.UNICODE)

def clean_topic_text(text: str) -> str:
    # Minimal cleaning: keep content, mostly remove stopwords/noise
    text = str(text).lower()
    text = url_pattern.sub(' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    tokens = token_pattern.findall(text)
    kept = []
    for tok in tokens:
        if tok in all_stopwords:
            continue
        if tok.isnumeric():
            continue
        if len(tok) < 2:
            continue
        kept.append(tok)

    return ' '.join(kept)

rt_topic_df['topic_text_clean'] = rt_topic_df['topic_text_raw'].apply(clean_topic_text)

# 5) Chunking (implemented): split long cleaned texts into overlapping token windows
CHUNK_SIZE_TOKENS = 220
CHUNK_OVERLAP_TOKENS = 40
MIN_CHUNK_TOKENS = 40

if CHUNK_OVERLAP_TOKENS >= CHUNK_SIZE_TOKENS:
    raise ValueError('CHUNK_OVERLAP_TOKENS must be smaller than CHUNK_SIZE_TOKENS')

def chunk_text(text: str) -> list:
    tokens = str(text).split()
    if not tokens:
        return []

    step = CHUNK_SIZE_TOKENS - CHUNK_OVERLAP_TOKENS
    chunks = []
    for i in range(0, len(tokens), step):
        chunk_tokens = tokens[i:i + CHUNK_SIZE_TOKENS]
        if len(chunk_tokens) < MIN_CHUNK_TOKENS:
            continue
        chunks.append(' '.join(chunk_tokens))

    # If article is short but non-empty, keep one chunk
    if not chunks and len(tokens) > 0:
        chunks = [' '.join(tokens)]

    return chunks

rt_topic_df['topic_text_model'] = rt_topic_df['topic_text_clean']
rt_topic_df['topic_chunks'] = rt_topic_df['topic_text_model'].apply(chunk_text)

# Build chunk-level modelling frame for BERTopic
model_df = rt_topic_df[['topic_date', 'topic_chunks']].explode('topic_chunks').rename(columns={'topic_chunks': 'topic_text_model'})
model_df = model_df[
    model_df['topic_date'].notna()
    & model_df['topic_text_model'].notna()
    & model_df['topic_text_model'].str.len().gt(0)
].copy()

# Full RT_de by default; set MAX_DOCS for quick local tests only
MAX_DOCS = None
if MAX_DOCS is not None and len(model_df) > MAX_DOCS:
    model_df = model_df.sample(MAX_DOCS, random_state=42)

model_df = model_df.sort_values('topic_date').reset_index(drop=True)

if model_df.empty:
    raise ValueError('No usable RT_de chunks after preprocessing.')

docs = model_df['topic_text_model'].tolist()
timestamps = model_df['topic_date'].dt.to_pydatetime().tolist()

print(f'RT_de articles used: {rt_topic_df["topic_date"].notna().sum()}')
print(f'RT_de chunks used for BERTopic: {len(docs)}')
print(f'Date range: {model_df["topic_date"].min().date()} to {model_df["topic_date"].max().date()}')

# Sanity check: most frequent tokens after cleaning
token_counts = Counter(' '.join(rt_topic_df['topic_text_clean']).split())
top_tokens_df = pd.DataFrame(token_counts.most_common(20), columns=['token', 'count'])
display(top_tokens_df)

# 6) BERTopic configuration (stable + reproducible)
topic_model = BERTopic(
    language='multilingual',
    embedding_model='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
    min_topic_size=25,
    calculate_probabilities=False,
    verbose=True,
)

topics, _ = topic_model.fit_transform(docs)
topic_info = topic_model.get_topic_info()

display(topic_info.head(20))
topic_model.visualize_barchart(top_n_topics=12)


HF_TOKEN not set. Continuing without explicit HF login.
RT_de articles used: 12390
RT_de chunks used for BERTopic: 25347
Date range: 2024-11-14 to 2026-01-19


,token,count
0,ukraine,28094
1,us,27029
2,russland,26916
3,trump,20534
4,usa,18611
5,eu,17386
6,russischen,14843
7,thema,14147
8,prozent,11785
9,präsident,10992


2026-02-19 16:54:04,959 - BERTopic - Embedding - Transforming documents to embeddings.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/793 [00:00<?, ?it/s]

2026-02-19 16:56:28,871 - BERTopic - Embedding - Completed ✓
2026-02-19 16:56:28,872 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-02-19 16:56:38,835 - BERTopic - Dimensionality - Completed ✓
2026-02-19 16:56:38,837 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-02-19 16:56:41,332 - BERTopic - Cluster - Completed ✓
2026-02-19 16:56:41,337 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-02-19 16:56:43,296 - BERTopic - Representation - Completed ✓


Topic  Count                                         Name  \
0      -1  13000                    -1_ukraine_russland_eu_us   
1       0   1162                        0_afd_spd_cdu_prozent   
2       1    770             1_israel_gaza_hamas_gazastreifen   
3       2    667                    2_venezuela_maduro_us_usa   
4       3    655              3_putin_trump_ukraine_präsident   
5       4    559         4_afd_deutschen_deutschland_deutsche   
6       5    551      5_selenskij_ukraine_selenskijs_wladimir   
7       6    472          6_prozent_euro_deutschland_deutsche   
8       7    422                      7_trump_musk_us_epstein   
9       8    363             8_china_chinesische_peking_zölle   
10      9    317                 9_ukraine_milliarden_eu_kiew   
11     10    304  10_streitkräfte_ukrainischen_truppen_gebiet   
12     11    270           11_europa_russland_europäischen_eu   
13     12    260     12_sanktionen_russland_eu_sanktionspaket   
14     13    223                13_gas_gazprom_lng_kubikmeter   
15     14    214       14_oreschnik_raketen_rakete_reichweite   
16     15    206      15_russischen_russische_russland_medien   
17     16    198                16_corona_covid_mrna_pandemie   
18     17    190              17_schiff_ostsee_schiffe_tanker   
19     18    160               18_polizei_verletzt_mann_täter   

                                                                                                           Representation  \
0                                       [ukraine, russland, eu, us, russischen, usa, nato, deutschland, thema, russische]   
1                                                   [afd, spd, cdu, prozent, bsw, partei, grünen, habeck, fpö, bundestag]   
2              [israel, gaza, hamas, gazastreifen, israelischen, netanjahu, israels, israelische, geiseln, palästinenser]   
3                      [venezuela, maduro, us, usa, venezuelas, venezolanischen, trump, karibik, venezolanische, caracas]   
4                                [putin, trump, ukraine, präsident, treffen, moskau, russland, us, präsidenten, wladimir]   
5                                         [afd, deutschen, deutschland, deutsche, berlin, spd, medien, cdu, thema, krieg]   
6                     [selenskij, ukraine, selenskijs, wladimir, ukrainischen, ukrainische, kiew, trump, präsident, nabu]   
7                 [prozent, euro, deutschland, deutsche, deutschen, merz, wirtschaft, milliarden, millionen, unternehmen]   
8                                          [trump, musk, us, epstein, donald, trumps, elon, präsident, biden, demokraten]   
9                                           [china, chinesische, peking, zölle, chinesischen, xi, usa, chinas, us, trump]   
10                                   [ukraine, milliarden, eu, kiew, euro, dollar, geld, unterstützung, us, ukrainischen]   
11           [streitkräfte, ukrainischen, truppen, gebiet, stadt, podoljaka, pokrowsk, sumy, ukrainische, frontabschnitt]   
12                                         [europa, russland, europäischen, eu, usa, europas, welt, krieg, nato, ukraine]   
13                         [sanktionen, russland, eu, sanktionspaket, trump, us, moskau, russischen, verhängt, präsident]   
14                        [gas, gazprom, lng, kubikmeter, pipeline, eu, russischem, gaslieferungen, slowakei, russisches]   
15               [oreschnik, raketen, rakete, reichweite, drohnen, systeme, einsatz, entwicklung, satelliten, kilometern]   
16  [russischen, russische, russland, medien, spionage, moskau, geheimdienste, journalisten, informationen, geheimdienst]   
17                                    [corona, covid, mrna, pandemie, who, impfstoffe, impfung, impfungen, epa, biontech]   
18                           [schiff, ostsee, schiffe, tanker, marine, schattenflotte, flagge, nato, finnischen, estland]   
19                               [polizei, verletzt, mann, täter, magdeburg, verletzte, jährige, auto, jähriger, mehrere]   

            

In [54]:
# Topics over time (RT_de)
topics_over_time = topic_model.topics_over_time(
    docs=docs,
    timestamps=timestamps,
    nr_bins=20,
)

display(topics_over_time.head(20))

topic_model.visualize_topics_over_time(
    topics_over_time,
    top_n_topics=10,
)


20it [00:47,  2.36s/it]


,Topic,Words,Frequency,Timestamp
0,-1,"ukraine, russland, nato, russischen, us",730,2024-11-13 13:39:21.600
1,0,"spd, bsw, afd, cdu, grünen",71,2024-11-13 13:39:21.600
2,1,"israel, netanjahu, israelischen, hamas, gaza",25,2024-11-13 13:39:21.600
3,2,"venezuela, peru, venezuelas, us, usa",7,2024-11-13 13:39:21.600
4,3,"ukraine, putin, trump, präsidenten, wladimir",23,2024-11-13 13:39:21.600
5,4,"rothschild, wanderwitz, deutschen, deutschland, krieg",28,2024-11-13 13:39:21.600
6,5,"selenskij, ukraine, selenskijs, trump, kiew",20,2024-11-13 13:39:21.600
7,6,"prozent, deutschland, deutschen, euro, ifo",23,2024-11-13 13:39:21.600
8,7,"trump, biden, bhattacharya, us, hunter",15,2024-11-13 13:39:21.600
9,8,"china, peking, chinesische, chinas, xi",26,2024-11-13 13:39:21.600
